# Lecture 16: Bagging Example

This notebook demonstrates the effect of bagging on decision tree performance using the mobility dataset.

In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

In [2]:
# Load mobility data
mob = pd.read_csv("mobility.csv")

# Prepare data (select relevant columns and handle missing values)
mob = (
    mob[mob["Mobility"].notnull()]
    .drop(["ID", "Name", "State"], axis=1)
    .astype(float)
    .fillna(-9000.0)
)

X = mob.drop("Mobility", axis=1).values
y = mob["Mobility"].values

print(f"Dataset size: {len(mob)} observations")
print(f"Target: Mobility")
print(f"X shape: {X.shape}, y shape: {y.shape}")

Dataset size: 729 observations
Target: Mobility
X shape: (729, 39), y shape: (729,)


In [3]:
# Perform 5-fold cross-validation for single tree
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

single_tree_mse_folds = []
for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    tree = DecisionTreeRegressor(random_state=42)
    tree.fit(X_train, y_train)
    preds = tree.predict(X_val)
    print(preds)
    print(y_val)
    mse = np.mean((y_val - preds)**2)
    single_tree_mse_folds.append(mse)

single_tree_mse = np.mean(single_tree_mse_folds)
print(f"Single tree 5-fold CV MSE: {single_tree_mse:.6f}")

[0.13065937 0.04496967 0.04282983 0.05183585 0.05476467 0.05355094
 0.04967466 0.04967466 0.04677287 0.08032332 0.06737538 0.04153846
 0.08134437 0.06850575 0.06389531 0.0327529  0.0327529  0.05781661
 0.08658836 0.12246586 0.0271325  0.04998663 0.05052314 0.08658836
 0.07105666 0.05166313 0.04532973 0.06243038 0.03360882 0.03933333
 0.05234302 0.06135577 0.05234302 0.0399361  0.04104875 0.05828221
 0.04978848 0.08545668 0.07340099 0.04900632 0.11693989 0.11693989
 0.05398111 0.07142857 0.09109904 0.07655502 0.07516681 0.11074919
 0.07516681 0.06461538 0.09630954 0.08545668 0.10495847 0.10360825
 0.08353081 0.08013305 0.11312765 0.09388971 0.08013305 0.04677287
 0.09630954 0.14209968 0.04677287 0.11485497 0.14215985 0.17888699
 0.09230257 0.07621671 0.06440809 0.08911483 0.05660012 0.10969793
 0.08524465 0.08134437 0.07621671 0.08134437 0.08145091 0.09230257
 0.12596899 0.08602151 0.07567567 0.10137276 0.18674137 0.18674137
 0.1691843  0.15948276 0.25056946 0.33830845 0.19205298 0.2533

In [4]:
# Perform 5-fold cross-validation for bagging
max_trees = 100
bag_mse_all_folds = []

print(f"Performing 5-fold CV for bagging with up to {max_trees} trees...")
for fold_num, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"  Fold {fold_num}/{n_folds}...")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Pre-compute all bagged trees for this fold
    trees = []
    for b in range(max_trees):
        # Bootstrap sample
        boot_idx = np.random.choice(len(X_train), size=len(X_train), replace=True)
        X_boot = X_train[boot_idx]
        y_boot = y_train[boot_idx]

        # Fit tree on bootstrap sample
        tree = DecisionTreeRegressor(random_state=b)
        tree.fit(X_boot, y_boot)
        trees.append(tree)

    # Compute predictions for all trees on validation set
    all_predictions = np.array([tree.predict(X_val) for tree in trees])

    # Compute MSE for each ensemble size
    fold_mse = []
    for num_trees in range(1, max_trees + 1):
        ensemble_preds = np.mean(all_predictions[:num_trees], axis=0)
        mse = np.mean((y_val - ensemble_preds)**2)
        fold_mse.append(mse)

    bag_mse_all_folds.append(fold_mse)

# Average MSE across folds for each ensemble size
bag_mse = np.mean(bag_mse_all_folds, axis=0)
print("Done!")

Performing 5-fold CV for bagging with up to 100 trees...
  Fold 1/5...


  Fold 2/5...


  Fold 3/5...


  Fold 4/5...


  Fold 5/5...


Done!
